In [1]:
import torch  # harus paling atas
import sys; sys.path.append("src")
import numpy as np
from collections import Counter
import retrieval as L

res = L.load_resources()                      # default: data/index/langchain/faiss.index
POS = {d.metadata["pasal"]: i for i, d in enumerate(res.docs)}

bm25_r = L.build_retriever(res, "bm25", top_k=5)
sem_r  = L.build_retriever(res, "semantic", top_k=5)

def ringkas(d, n=90):
    isi = d.page_content.split("\n", 1)[-1].replace("\n", " ")
    return isi[:n] + ("..." if len(isi) > n else "")

def tampil(docs):
    for r, d in enumerate(docs, 1):
        print(f"  {r}. Pasal {d.metadata['pasal']:<4} {d.metadata['score']:.4f}  {ringkas(d)}")

# BM25

In [2]:
def kontribusi(i, tok):
    """Skor BM25 pasal ke-i dipecah per kata query (jumlahnya = skor get_scores)."""
    b, dl, out = res.bm25, res.bm25.doc_len[i], {}
    for t, n in Counter(tok).items():
        f = b.doc_freqs[i].get(t, 0)
        if f:
            out[t] = n * (b.idf.get(t) or 0) * f * (b.k1 + 1) / (f + b.k1 * (1 - b.b + b.b * dl / b.avgdl))
    return out

def bm25_detail(q):
    tok = L.tokenize(q)
    uniq = list(dict.fromkeys(tok))
    print(f"### {q}\ntoken            : {tok}")
    print("tidak ada di korpus:", [t for t in uniq if t not in res.bm25.idf] or "-")
    for t in uniq:
        if t in res.bm25.idf:
            df = sum(1 for f in res.bm25.doc_freqs if t in f)
            print(f"  {t:<14} df={df:<4} idf={res.bm25.idf[t]:.2f}")
    print("top-5:")
    for r, d in enumerate(bm25_r.invoke(q), 1):
        kon = sorted(kontribusi(POS[d.metadata["pasal"]], tok).items(), key=lambda x: -x[1])[:4]
        print(f"  {r}. Pasal {d.metadata['pasal']:<4} {d.metadata['score']:.2f}  <- " + ", ".join(f"{t} {s:.1f}" for t, s in kon))

bm25_detail("nyolong motor tetangga itu pasal apa")

### nyolong motor tetangga itu pasal apa
token            : ['nyolong', 'motor', 'tetangga', 'itu', 'pasal', 'apa']
tidak ada di korpus: ['nyolong', 'motor']
  tetangga       df=1    idf=6.03
  itu            df=35   idf=2.81
  pasal          df=624  idf=1.31
  apa            df=14   idf=3.74
top-5:
  1. Pasal 265  9.68  <- tetangga 8.0, pasal 1.7
  2. Pasal 209  6.66  <- apa 3.8, pasal 2.9
  3. Pasal 174  6.31  <- apa 4.7, pasal 1.6
  4. Pasal 332  6.20  <- apa 5.2, pasal 1.0
  5. Pasal 131  6.19  <- itu 4.0, pasal 2.2


In [3]:
bm25_detail("pencurian dengan kekerasan")

### pencurian dengan kekerasan
token            : ['pencurian', 'dengan', 'kekerasan']
tidak ada di korpus: -
  pencurian      df=6    idf=4.56
  dengan         df=505  idf=1.31
  kekerasan      df=41   idf=2.64
top-5:
  1. Pasal 245  11.50  <- pencurian 6.2, kekerasan 3.6, dengan 1.8
  2. Pasal 477  9.74  <- pencurian 7.9, dengan 1.8
  3. Pasal 479  9.65  <- pencurian 5.1, kekerasan 2.3, dengan 2.2
  4. Pasal 476  7.99  <- pencurian 5.8, dengan 2.2
  5. Pasal 414  7.01  <- kekerasan 4.5, dengan 2.5


# SEMANTIC



In [4]:
def semantic_detail(q):
    print(f"### {q}")
    tampil(sem_r.invoke(q))

semantic_detail("nyolong motor tetangga itu pasal apa")

### nyolong motor tetangga itu pasal apa


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  1. Pasal 476  0.5245   Setiap Orang yang mengambil suatu Barang yang sebagian atau seluruhnya milik orang lain, ...
  2. Pasal 486  0.5235  Setiap Orang yang secara melawan hukum memiliki suatu Barang yang sebagian atau seluruhnya...
  3. Pasal 481  0.5136  (1) Penuntutan pidana tidak dilakukan jika yang melakukan salah satu Tindak Pidana sebagai...
  4. Pasal 478  0.5100   Jika Tindak Pidana sebagaimana dimaksud dalam Pasal 476 dan Pasal 477 ayat (1) huruf f da...
  5. Pasal 483  0.5097   (1) Dipidana karena pengancaman dengan pidana penjara paling lama 4 (empat) tahun atau pi...


In [5]:
import faiss
V = res.index.reconstruct_n(0, res.n)
n_ = np.linalg.norm(V, axis=1)
print("vektor:", res.index.ntotal, "| dim:", res.index.d, "| norma min/maks:", n_.min().round(4), n_.max().round(4),
      "| NaN:", int(np.isnan(V).sum()))

def tetangga(p, k=5):
    D, I = res.index.search(V[POS[p]][None], k + 1)
    print(f"\nTetangga terdekat Pasal {p}: {ringkas(res.docs[POS[p]], 70)}")
    for j, s in zip(I[0][1:], D[0][1:]):
        print(f"  Pasal {res.docs[j].metadata['pasal']:<4} {s:.4f}  {ringkas(res.docs[j])}")

for p in (476, 34):
    tetangga(p)

vektor: 624 | dim: 1024 | norma min/maks: 1.0 1.0 | NaN: 0

Tetangga terdekat Pasal 476:  Setiap Orang yang mengambil suatu Barang yang sebagian atau seluruhny...
  Pasal 486  0.8637  Setiap Orang yang secara melawan hukum memiliki suatu Barang yang sebagian atau seluruhnya...
  Pasal 477  0.8499   (1) Dipidana dengan pidana penjara paling lama 7 (tujuh) tahun atau pidana denda paling b...
  Pasal 479  0.8479   (1) Setiap Orang yang melakukan pencurian yang didahului, disertai, atau diikuti dengan K...
  Pasal 497  0.8375   Setiap Orang yang menjadikan sebagai mata pencaharian atau kebiasaan membeli Barang denga...
  Pasal 496  0.8331   Setiap Orang yang memperoleh secara curang suatu jasa untuk diri sendiri atau orang lain ...

Tetangga terdekat Pasal 34:  Setiap Orang yang terpaksa melakukan perbuatan yang dilarang tidak di...
  Pasal 43   0.7970  Setiap Orang yang melakukan pembelaan terpaksa yang melampaui batas yang langsung disebabk...
  Pasal 33   0.7666   Setiap Orang yang mela

# BM25 VS SEMANTIC


In [6]:
daftar = [
    "nyolong motor tetangga itu pasal apa",
    "ngerampas hp orang pakai todongan pisau",
    "nipu orang jual beli online pura-pura jual barang",
    "aku disiul siulkan oleh orang random",
    "pencurian dengan kekerasan",
]

def banding(q):
    b = [d.metadata["pasal"] for d in bm25_r.invoke(q)]
    s = [d.metadata["pasal"] for d in sem_r.invoke(q)]
    print(f"\n### {q}\n  BM25     : {b}\n  semantic : {s}\n  irisan   : {sorted(set(b) & set(s)) or '-'}")

for q in daftar:
    banding(q)


### nyolong motor tetangga itu pasal apa
  BM25     : [265, 209, 174, 332, 131]
  semantic : [476, 486, 481, 478, 483]
  irisan   : -

### ngerampas hp orang pakai todongan pisau
  BM25     : [520, 451, 530, 145, 450]
  semantic : [461, 479, 259, 476, 458]
  irisan   : -

### nipu orang jual beli online pura-pura jual barang
  BM25     : [273, 520, 543, 489, 497]
  semantic : [493, 389, 492, 387, 506]
  irisan   : -

### aku disiul siulkan oleh orang random
  BM25     : [37, 42, 25, 489, 149]
  semantic : [43, 109, 167, 331, 354]
  irisan   : -

### pencurian dengan kekerasan
  BM25     : [245, 477, 479, 476, 414]
  semantic : [479, 476, 542, 450, 478]
  irisan   : [476, 479]


# + QUERY REWRITE & EXPAND

In [8]:
# # VARIAN = "v3_role_context_fewshot"
# VARIAN = "v1_fewshot"

# def bentuk_query(q):
#     r = L.rewrite_query(q, VARIAN)
#     return {"asli": q, "rewrite": r, "expand": L.expand_query(q, VARIAN), "rewrite+expand": L.expand_query(r, VARIAN)}

# def top5(retr, q):
#     return [d.metadata["pasal"] for d in retr.invoke(q)]

# for q in daftar[:3]:
#     print(f"\n### {q}")
#     for nama, qq in bentuk_query(q).items():
#         print(f"[{nama}] {qq}")
#         print(f"   BM25     : {top5(bm25_r, qq)}")
#         print(f"   semantic : {top5(sem_r, qq)}")

In [9]:
VARIAN = "v0_baseline"
q_chain = L.build_query_chain(use_rewriting=True, use_expansion=True, variant=VARIAN)
# q_chain = L.build_query_chain(use_rewriting=True, variant=VARIAN)

for q in daftar[4:]:
    qq = q_chain.invoke(q)
    print(f"\n### {q}\n  query akhir: {qq}")
    print("  BM25     :", [d.metadata["pasal"] for d in bm25_r.invoke(qq)])
    print("  semantic :", [d.metadata["pasal"] for d in sem_r.invoke(qq)])


### pencurian dengan kekerasan
  query akhir: Tindak pidana pencurian yang dilakukan dengan disertai atau disertai dengan kekerasan atau ancaman kekerasan terhadap orang. Perampokan
  BM25     : [479, 543, 414, 303, 477]
  semantic : [479, 476, 156, 12, 458]


In [ ]:
VARIAN = "v0_baseline"
q_chain = L.build_query_chain(use_rewriting=True, use_expansion=True, variant=VARIAN)
# q_chain = L.build_query_chain(use_rewriting=True, variant=VARIAN)

for q in daftar[0]:
    qq = q_chain.invoke(q)
    print(f"\n### {q}\n  query akhir: {qq}")
    print("  BM25     :", [d.metadata["pasal"] for d in bm25_r.invoke(qq)])
    print("  semantic :", [d.metadata["pasal"] for d in sem_r.invoke(qq)])